In [ ]:
%%sql -r dataframe_1
USE DATABASE FINANCE_DB;
USE SCHEMA FINANCE;

In [ ]:
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark.types import StructType, StructField, StringType

session = get_active_session()
session.use_role('ACCOUNTADMIN')
session.use_database('FINANCE_DB')
session.use_schema('FINANCE')

In [ ]:
schema = StructType([
    StructField('TYPE', StringType()),
    StructField('PRODUCT', StringType()),
    StructField('STARTED_DATE', StringType()),
    StructField('COMPLETED_DATE', StringType()),
    StructField('DESCRIPTION', StringType()),
    StructField('AMOUNT', StringType()),
    StructField('FEE', StringType()),
    StructField('CURRENCY', StringType()),
    StructField('STATE', StringType()),
    StructField('BALANCE', StringType()),
])

df = session.read.schema(schema).option('skip_header', 1).option('field_optionally_enclosed_by', '"').csv('@finance_sources/Revolut/')
df.show(15)

In [ ]:
table_name = 'FINANCE_DB.FINANCE.REVOLUT_TRANSACTIONS'

existing_tables = [row['name'] for row in session.sql("SHOW TABLES IN FINANCE_DB.FINANCE").collect()]

if 'REVOLUT_TRANSACTIONS' not in existing_tables:
    df.write.mode('overwrite').save_as_table(table_name)
    print(f'Created table and saved {df.count()} rows to {table_name}')
else:
    existing_df = session.table(table_name)
    join_cols = ['STARTED_DATE', 'DESCRIPTION', 'AMOUNT', 'CURRENCY']
    new_rows = df.join(existing_df, on=join_cols, how='left_anti')
    count = new_rows.count()
    if count > 0:
        new_rows.write.mode('append').save_as_table(table_name)
        print(f'Appended {count} new rows to {table_name}')
    else:
        print(f'No new rows to append — all rows already exist in {table_name}')

In [ ]:
%%sql -r dataframe_3
CREATE OR REPLACE VIEW FINANCE_DB.FINANCE.REVOLUT_TRANSACTIONS_VW AS
SELECT
    TO_DATE(STARTED_DATE, 'YYYY-MM-DD HH24:MI:SS') AS TRANSACTION_DATE,
    TYPE,
    DESCRIPTION,
    AMOUNT::NUMBER(10,2) AS AMOUNT,
    FEE::NUMBER(10,2) AS FEE,
    CURRENCY,
    STATE,
    BALANCE::NUMBER(10,2) AS BALANCE
FROM FINANCE_DB.FINANCE.REVOLUT_TRANSACTIONS

In [ ]:
%%sql -r dataframe_2
SELECT DATE_TRUNC('month', TRANSACTION_DATE) AS MONTH, 
        count(1) as TRAN_CNT,
        SUM(TRY_CAST(AMOUNT AS NUMBER(10,2))) AS TOTAL_LOCAL_AMOUNT
FROM FINANCE_DB.FINANCE.REVOLUT_TRANSACTIONS_VW
GROUP BY MONTH
ORDER BY MONTH;